In [18]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/angelarivass/Angela_MineriaDeDatos/refs/heads/main/googleplaystore.csv"
 
df = pd.read_csv(url)

RETO 1: La Trampa de las Unidades de Medida

Haciendo una función para obtener únicamente el valor numérico en los registros de la columna size. 

In [19]:
df['Size'].head()

0     19M
1     14M
2    8.7M
3     25M
4    2.8M
Name: Size, dtype: object

Los registros contienen el caracter 'M' o 'k', y tambien encontre un registro '1000+' por eso hacemos la siguiente funcion:

In [ ]:
def depurarNum(texto):
    texto = str(texto).strip()

    terminacion_M = texto.endswith("M")
    terminacion_k = texto.endswith("k")

    if terminacion_M:
        return float(texto.removesuffix("M"))

    if terminacion_k:
        return float(texto.removesuffix("k")) / 1024
    
    if texto == "Varies with device":
        return np.nan

    else:
        try:
            return float(texto)
        except ValueError:  #para el caso de '1000+'
            return np.nan

Aplicamos la función

In [43]:
df['size_clean'] = df['Size'].apply(depurarNum)

Vemos los primeros 10 registros para confirmar que solo sean valores numericos

In [44]:
df['size_clean'].head(10)


0    19.0
1    14.0
2     8.7
3    25.0
4     2.8
5     5.6
6    19.0
7    29.0
8    33.0
9     3.1
Name: size_clean, dtype: float64

Ya no tienen la terminacion en M o k

Instruccion: Una vez convertida la columna a valores numéricos (Megabytes), ejecutar el método .mean(). 

--> Fue en este paso donde encontre el refistro de '1000+'

In [ ]:
promedio = df['size_clean'].mean()
print(f'Promedio: {promedio}')

Promedio: 21.51616543577433


--> El peso promedio en Megabytes de las apps en la play store es de 21.5161 Megabytes

RETO 2: El Tipo de Dato Cronológico

Usando la funcion pd.to_datetime()

In [65]:
df['Last Updated'].head()

0     January 7, 2018
1    January 15, 2018
2      August 1, 2018
3        June 8, 2018
4       June 20, 2018
Name: Last Updated, dtype: object

Se identifico un registro con 1.0.19, eliminar

In [71]:
df['Last Updated_clean'] = df['Last Updated']

df['Last Updated_clean'] = df['Last Updated_clean'].replace("1.0.19", "")

df['Last Updated_clean'] = pd.to_datetime(df['Last Updated_clean'])

In [72]:
df['Year_updated'] = df['Last Updated_clean'].dt.year

Pregunta: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

In [73]:
df['Year_updated'].value_counts()

Year_updated
2018.0    7349
2017.0    1867
2016.0     804
2015.0     459
2014.0     209
2013.0     110
2012.0      26
2011.0      15
2010.0       1
Name: count, dtype: int64

--> El año en que se actualizaron más apps fue en 2018, con 7349 actualizaciones

RETO 3: La Decisión Arquitectónica
¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?

In [76]:
df['size_clean'].isnull().sum()

np.int64(1696)

Hay 1696 registros con valores NaN (nulos)
--> Debido a la cantidad de elementos nulos en los registros, yo decidiría eliminarlos con la funcion dropna(). Esto porque se trata de una gran cantidad de registros los que se estarían llenando con predicciones y no datos reales. A pesar de que sean aproximaciones basadas en datos reales, opino que sería mejor decisión trabajar con el conjunto original cuando el peso es una variable crítica, esro para mantener la distribución del peso sin sesgar resultados.